# Gold Layer - Fact Sales

## Objective

This notebook creates the Gold Fact Sales table by integrating
validated Silver tables.

The objective is to provide a centralized business-ready dataset
that supports analytics, reporting and dashboarding.

### Source Tables

- silver.orders
- silver.order_items
- silver.products
- silver.customers
- silver.payments

### Target Table

gold.fact_sales

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
RUN_ID = generate_run_id()
START_TIME = start_pipeline()
PIPELINE_NAME = "Gold_Fact_Sales_Load"
TARGET_TABLE = "retailmart.gold.fact_sales"

Pipeline Started : 2026-07-19 04:31:10.861965


In [0]:
print("GOLD FACT SALES PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD FACT SALES PIPELINE
Pipeline : Gold_Fact_Sales_Load
Run ID : 4e3aaf20-202e-4fc3-ba1e-d7bd60b78851
Target : retailmart.gold.fact_sales


In [0]:
spark.sql(f"""

CREATE OR REPLACE TABLE {TARGET_TABLE} AS

WITH payment_summary AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value,
        MAX(payment_installments) AS payment_installments,

        CASE
            WHEN COUNT(DISTINCT payment_type) = 1
            THEN MAX(payment_type)
            ELSE 'multiple'
        END AS payment_type

    FROM {SILVER_PAYMENTS}
    GROUP BY order_id

)

SELECT

    -- Order Details

    oi.order_id,
    oi.order_item_id,

    o.order_status,
    o.order_purchase_timestamp,
    YEAR(o.order_purchase_timestamp) AS order_year,
    MONTH(o.order_purchase_timestamp) AS order_month,
    DATE_FORMAT(
    o.order_purchase_timestamp,
    'yyyy-MM'
    ) AS revenue_month,
    o.order_delivered_customer_date,
    o.delivery_duration_days,

    -- Customer Details

    o.customer_id,
    c.customer_city,
    c.customer_state,

    -- Product Details

    oi.product_id,
    p.product_category_name,

    -- Sales Metrics

    oi.price,
    oi.freight_value,

    ROUND(
        oi.price + oi.freight_value,
        2
    ) AS total_item_value,

    -- Payment Details

    ps.payment_type,
    ps.payment_installments,
    ps.total_payment_value

FROM {SILVER_ORDER_ITEMS} oi

INNER JOIN {SILVER_ORDERS} o
    ON oi.order_id = o.order_id

INNER JOIN {SILVER_PRODUCTS} p
    ON oi.product_id = p.product_id

INNER JOIN {SILVER_CUSTOMERS} c
    ON o.customer_id = c.customer_id

LEFT JOIN payment_summary ps
    ON oi.order_id = ps.order_id

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
assert spark.catalog.tableExists(TARGET_TABLE), "Gold table creation failed!"
print("Gold Fact Sales table created successfully.")

Gold Fact Sales table created successfully.


In [0]:
fact_sales_df = spark.table(TARGET_TABLE)

print("GOLD FACT SALES VALIDATION SUMMARY")
print(f"Rows Written      : {fact_sales_df.count()}")
print(f"Unique Orders     : {fact_sales_df.select('order_id').distinct().count()}")
print(f"Unique Customers  : {fact_sales_df.select('customer_id').distinct().count()}")
print(f"Unique Products   : {fact_sales_df.select('product_id').distinct().count()}")
print(f"Target Table      : {TARGET_TABLE}")

GOLD FACT SALES VALIDATION SUMMARY
Rows Written      : 86328
Unique Orders     : 50000
Unique Customers  : 14481
Unique Products   : 3000
Target Table      : retailmart.gold.fact_sales


In [0]:
# Verify 
fact_sales_df = spark.table(TARGET_TABLE)
rows_written = fact_sales_df.count()

print(f"Rows in Fact Sales : {rows_written}")
fact_sales_df.printSchema()
display(fact_sales_df.limit(10))

Rows in Fact Sales : 86328
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- total_payment_value: double (nullable = true)



order_id,order_item_id,order_status,order_purchase_timestamp,order_year,order_month,revenue_month,order_delivered_customer_date,delivery_duration_days,customer_id,customer_city,customer_state,product_id,product_category_name,price,freight_value,total_item_value,payment_type,payment_installments,total_payment_value
ORD_0000001,1,delivered,2023-06-15T14:30:00.000Z,2023,6,2023-06,2023-06-19T14:30:00.000Z,4,CUST_006571,Belo Horizonte,MG,PROD_001950,books,1754.7,20.17,1774.87,multiple,12,2807.85
ORD_0000002,1,cancelled,2021-06-12T11:02:00.000Z,2021,6,2021-06,null,null,CUST_006956,Aracaju,SE,PROD_000989,food,2105.2,45.48,2150.68,voucher,1,805.73
ORD_0000003,1,delivered,2022-03-31T19:38:00.000Z,2022,3,2022-03,2022-04-14T19:38:00.000Z,14,CUST_008373,Joao Pessoa,PB,PROD_000254,furniture,1627.94,78.29,1706.23,credit_card,6,508.88
ORD_0000004,1,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001327,music,65.22,31.08,96.3,boleto,3,1624.71
ORD_0000004,2,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_002714,computers,946.39,44.43,990.82,boleto,3,1624.71
ORD_0000004,3,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001507,home_appliances,157.96,55.93,213.89,boleto,3,1624.71
ORD_0000005,1,shipped,2022-08-27T07:46:00.000Z,2022,8,2022-08,null,null,CUST_002810,Manaus,AM,PROD_001872,fashion,1681.83,14.0,1695.83,credit_card,3,1592.91
ORD_0000006,1,invoiced,2023-05-26T16:27:00.000Z,2023,5,2023-05,null,null,CUST_007867,Recife,PE,PROD_001518,garden,856.74,64.83,921.57,credit_card,1,495.97
ORD_0000007,1,processing,2021-09-21T15:50:00.000Z,2021,9,2021-09,null,null,CUST_004827,Campo Grande,MS,PROD_001504,music,373.8,66.88,440.68,credit_card,2,1738.92
ORD_0000008,1,delivered,2021-01-19T14:24:00.000Z,2021,1,2021-01,2021-01-28T14:24:00.000Z,9,CUST_005137,Porto Velho,RO,PROD_001133,health,899.98,23.95,923.93,voucher,1,1107.69


In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS payment_count
FROM retailmart.silver.payments
GROUP BY order_id
HAVING COUNT(*) > 1
LIMIT 10;

order_id,payment_count
ORD_0000001,2
ORD_0000004,2
ORD_0000012,2
ORD_0000023,2
ORD_0000025,2
ORD_0000038,2
ORD_0000041,2
ORD_0000052,2
ORD_0000057,2
ORD_0000061,2


In [0]:
%sql
SELECT
    order_id,
    order_item_id,
    COUNT(*) AS cnt
FROM retailmart.gold.fact_sales
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

order_id,order_item_id,cnt


In [0]:
%sql
SELECT COUNT(*) FROM retailmart.silver.order_items;

COUNT(*)
86328


In [0]:
%sql
SELECT COUNT(*) FROM retailmart.gold.fact_sales;

COUNT(*)
86328


In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source="Silver (Orders, Order Items, Products, Payments)",
    target=TARGET_TABLE,
    rows_read=None,
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS",
)

LOAD REPORT
Pipeline        : Gold_Fact_Sales_Load
Run ID          : 4e3aaf20-202e-4fc3-ba1e-d7bd60b78851
Source          : Silver (Orders, Order Items, Products, Payments)
Target          : retailmart.gold.fact_sales
Rows Read       : None
Rows Written    : 86328
Duplicate Rows  : 0
Start Time      : 2026-07-19 04:31:10.861965
End Time        : 2026-07-19 04:31:53.493766
Duration (sec)  : 42.63
Status          : SUCCESS


In [0]:
from datetime import datetime
END_TIME = datetime.now()
print(f"Execution Time : {END_TIME - START_TIME}")

Execution Time : 0:00:42.867615


## Engineering Observations

- The Gold Fact Sales table serves as the centralized analytical dataset for RetailMart and is designed to support business intelligence and reporting use cases.

- Data from validated Silver layer tables (Orders, Order Items, Customers, Products, and Payments) is integrated using SQL JOIN operations to create a unified view of sales transactions.

- Business metrics such as **Total Item Value** are derived during the transformation process, reducing computation overhead for downstream analytical queries.

- The Gold layer follows a denormalized star-schema style approach, making it suitable for dashboards, ad-hoc SQL analysis, and reporting tools.

- Spark SQL was used to perform transformations, demonstrating relational data processing concepts such as SELECT statements, JOINs, derived columns, and data aggregation readiness.

- The resulting Fact Sales table acts as the single source of truth for downstream Gold analytical datasets such as Monthly Revenue, Customer Segmentation, Product Ranking, and Sales Funnel Analysis.

- By separating Bronze, Silver, and Gold layers, the Medallion Architecture improves data quality, maintainability, scalability, and overall pipeline reliability while supporting production-style data engineering practices.